# Calcul de la duration de liquidité

Ce carnet reprend et structure le script `liquidity_duration.py` en cellules séparées pour faciliter l'exploration, l'exécution pas-à-pas et la documentation des étapes.

In [ ]:
# --- Imports et définitions initiales ---
import numpy as np
import pandas as pd
import json

# Portefeuille synthétique
PORTEFEUILLE = pd.DataFrame([
    ("Obligations souveraines zone euro",     0.19,  "ACPR (2025b), source"),
    ("Obligations dentreprises",             0.37,  "ACPR (2025b), source"),
    ("Actions et participations",             0.22,  "ACPR (2025b), source"),
    ("OPC non ventiles",                      0.16,  "ACPR (2025b), source"),
    ("Immobilier",                            0.04,  "Hypothese calibree"),
    ("Tresorerie et depots",                  0.02,  "Hypothese calibree"),
], columns=["classe", "poids", "origine"])

assert abs(PORTEFEUILLE["poids"].sum() - 1.0) < 1e-9, "Le portefeuille doit sommer a 100%"

TOTAL_PLACEMENTS_MDS_EUR = 2672.0  # SOURCE : ACPR n°173, fin decembre 2024

In [ ]:
# --- Calibration microstructure par classe d'actif ---
CALIBRATION = pd.DataFrame([
    ("Obligations souveraines zone euro",   3,        0.60,       0.50,    1.2),
    ("Obligations dentreprises",          18,        0.20,       0.55,    2.5),
    ("Actions et participations",           6,        0.90,       0.50,    1.0),
    ("OPC non ventiles",                   25,        0.15,       0.60,    3.5),
    ("Immobilier",                        150,        0.02,       0.85,   12.0),
    ("Tresorerie et depots",                1,       50.00,       0.30,    0.05),
], columns=["classe", "c_spread_bp", "V_pct_jour", "gamma", "lambda_pct"])

PI_PARTICIPATION = 0.12   # HYPOTHESE : taux de participation maximal au marche (12%)

print("Portefeuille synthetique charge :", len(PORTEFEUILLE), "classes")
print("Calibration microstructure chargee :", len(CALIBRATION), "classes")

# Optionnel : sauvegarder les tables si besoin
PORTEFEUILLE.to_csv("portefeuille.csv", index=False)
CALIBRATION.to_csv("calibration.csv", index=False)

In [ ]:
# --- Scenarios de stress ---
SCENARIOS = pd.DataFrame([
    ("Central",              0,             0,           1.0,           1.00),
    ("Choc de taux",       200,             0,           1.5,           0.85),
    ("Crise de spread",      0,           150,           1.3,           0.70),
    ("Choc combine",       200,           150,           2.0,           0.55),
], columns=["scenario", "choc_taux_pb", "choc_spread_pb", "mult_rachat", "prof_marche"])

TAUX_RACHAT_BASE = 0.047
MASS_LAPSE_SCR = 0.40

TAUX_ABSORPTION = {"Central": 0.50, "Choc de taux": 0.35,
                  "Crise de spread": 0.40, "Choc combine": 0.20,
                  "Sim1 - Effet direct (marche seul, hausse)": 0.50,
                  "Sim2 - Effet volume seul (rachats, hausse)": 0.35}

In [ ]:
# --- Fonction de coût de cession ---
def cout_cession(c_spread_bp, V_pct_jour, gamma, lambda_pct, Q_rel, h, pi=PI_PARTICIPATION):
    """
    Cout relatif de cession C_i(h) pour la classe i.
    """
    c_spread = c_spread_bp / 10_000.0
    if Q_rel <= 1e-12:
        return c_spread
    jours_necessaires = Q_rel / (pi * V_pct_jour / 100.0)
    ratio_urgence = jours_necessaires / h
    impact = (lambda_pct / 100.0) * (ratio_urgence ** gamma)
    return c_spread + impact

In [ ]:
# --- Ordre de liquidation et allocation du besoin ---
ORDRE_LIQUIDATION = [
    "Tresorerie et depots",
    "Obligations souveraines zone euro",
    "Actions et participations",
    "Obligations dentreprises",
    "OPC non ventiles",
    "Immobilier",
]

def allouer_besoin_liquidite(besoin_total_pct_portefeuille, poids):
    reste = besoin_total_pct_portefeuille
    alloc = {c: 0.0 for c in poids.index}
    for classe in ORDRE_LIQUIDATION:
        if reste <= 1e-12:
            break
        disponible = poids[classe]
        prelevement = min(reste, disponible)
        alloc[classe] = prelevement
        reste -= prelevement
    return alloc, reste

# Sensibilite de spread par classe
SENSIBILITE_SPREAD = {
    "Obligations souveraines zone euro": 0.0,
    "Obligations dentreprises":         1.0,
    "Actions et participations":         0.0,
    "OPC non ventiles":                  0.5,
    "Immobilier":                        0.0,
    "Tresorerie et depots":              0.0,
}

In [ ]:
print("
Modele de cout de cession et ordre de liquidation charges.")

# --- Calcul du besoin net de liquidite par scenario ---
def besoin_net_liquidite(scenario_row):
    nom = scenario_row["scenario"]
    rachat_brut = TAUX_RACHAT_BASE * scenario_row["mult_rachat"]
    absorption = TAUX_ABSORPTION[nom]
    return rachat_brut * (1 - absorption)

BESOINS = SCENARIOS.copy()
BESOINS["besoin_net_pct"] = BESOINS.apply(besoin_net_liquidite, axis=1)
print("
Besoin net de liquidite par scenario (% du portefeuille total) :")
print(BESOINS[["scenario", "besoin_net_pct"]].to_string(index=False,
      formatters={"besoin_net_pct": "{:.3%}".format}))

In [ ]:
# --- Coût de cession du portefeuille C_P(h) par scenario ---
HORIZONS = np.array([1, 2, 3, 5, 8, 10, 15, 20, 30, 45, 60, 90, 120])

poids_idx = PORTEFEUILLE.set_index("classe")["poids"]
calib_idx = CALIBRATION.set_index("classe")

def cout_portefeuille(scenario_row, h):
    besoin = besoin_net_liquidite(scenario_row)
    alloc, reste_non_couvert = allouer_besoin_liquidite(besoin, poids_idx)

    cout_total = 0.0
    detail = {}
    for classe in poids_idx.index:
        w_i = poids_idx[classe]
        q_i_total_pct = alloc[classe]
        q_i_rel = q_i_total_pct / w_i if w_i > 0 else 0.0

        row = calib_idx.loc[classe]
        c_spread_bp = row["c_spread_bp"] + (
            scenario_row["choc_spread_pb"] * SENSIBILITE_SPREAD[classe]
        )
        V_i = row["V_pct_jour"] * scenario_row["prof_marche"]
        V_i = max(V_i, 1e-6)

        c_i = cout_cession(c_spread_bp, V_i, row["gamma"], row["lambda_pct"], q_i_rel, h)
        detail[classe] = {"q_i_rel": q_i_rel, "c_i": c_i, "w_i": w_i}
        cout_total += w_i * c_i

    return cout_total, detail, reste_non_couvert

# Construction de la surface horizon x scenario
lignes = []
for _, scen in SCENARIOS.iterrows():
    for h in HORIZONS:
        c_p, detail, reste = cout_portefeuille(scen, h)
        lignes.append({
            "scenario": scen["scenario"], "horizon_j": h,
            "cout_portefeuille": c_p, "besoin_non_couvert": reste,
        })
SURFACE = pd.DataFrame(lignes)
SURFACE.to_csv("surface_horizon_scenario.csv", index=False)
print("
Surface horizon x scenario calculee :", len(SURFACE), "points")

In [ ]:
# --- Scenarios de decomposition et scenario baisse (pour duration) ---
SCENARIOS_DECOMP = pd.DataFrame([
    ("Sim1 - Effet direct (marche seul, hausse)",     200, 0, 1.0, 0.85),
    ("Sim2 - Effet volume seul (rachats, hausse)",     200, 0, 1.5, 1.00),
], columns=["scenario", "choc_taux_pb", "choc_spread_pb", "mult_rachat", "prof_marche"])

SCENARIOS_BAISSE = pd.DataFrame([
    ("Choc de taux (baisse symétrique)",              -200, 0, 0.5, 1.15),
    ("Sim1 - Effet direct (marche seul, baisse)",      -200, 0, 1.0, 1.15),
    ("Sim2 - Effet volume seul (rachats, baisse)",     -200, 0, 0.5, 1.00),
], columns=["scenario", "choc_taux_pb", "choc_spread_pb", "mult_rachat", "prof_marche"])

for nom in SCENARIOS_BAISSE["scenario"]:
    TAUX_ABSORPTION[nom] = 0.50

DELTA_R = 0.02

def _trouver_scenario(nom):
    for table in (SCENARIOS, SCENARIOS_DECOMP, SCENARIOS_BAISSE):
        if nom in table["scenario"].values:
            return table[table["scenario"] == nom].iloc[0]
    raise KeyError(nom)

def duration_liquidite_centree(nom_scenario_hausse, nom_scenario_baisse, horizons=HORIZONS):
    scen_central = SCENARIOS[SCENARIOS["scenario"] == "Central"].iloc[0]
    scen_up = _trouver_scenario(nom_scenario_hausse)
    scen_down = _trouver_scenario(nom_scenario_baisse)

    resultats = []
    for h in horizons:
        c_central, _, _ = cout_portefeuille(scen_central, h)
        c_up, _, _ = cout_portefeuille(scen_up, h)
        c_down, _, _ = cout_portefeuille(scen_down, h)
        d_l = (c_up - c_down) / (2 * DELTA_R) / c_central
        resultats.append({"horizon_j": h, "C_central": c_central,
                           "C_up": c_up, "C_down": c_down, "D_L": d_l})
    return pd.DataFrame(resultats)

DURATION_TOTALE = duration_liquidite_centree("Choc de taux", "Choc de taux (baisse symétrique)")
DURATION_DIRECTE = duration_liquidite_centree("Sim1 - Effet direct (marche seul, hausse)", "Sim1 - Effet direct (marche seul, baisse)")
DURATION_VOLUME = duration_liquidite_centree("Sim2 - Effet volume seul (rachats, hausse)", "Sim2 - Effet volume seul (rachats, baisse)")

DURATION_TOTALE.to_csv("duration_totale.csv", index=False)
DURATION_DIRECTE.to_csv("duration_directe.csv", index=False)
DURATION_VOLUME.to_csv("duration_volume.csv", index=False)

print("
Duration de liquidite totale D_L^r(h) -- difference finie CENTREE")

In [ ]:
# --- Figures ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "font.family": "serif", "font.size": 10.5,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150,
})

COULEURS_SCEN = {
    "Central": "#2c3e50", "Choc de taux": "#c0392b",
    "Crise de spread": "#d68910", "Choc combine": "#7d3c98",
}

# Figures (reprennent le code du script original)
h_fin = np.linspace(1, 120, 200)
Q_REF_ILLUSTRATION = 0.02
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for classe in poids_idx.index:
    row = calib_idx.loc[classe]
    couts = [cout_cession(row["c_spread_bp"], row["V_pct_jour"], row["gamma"],
                           row["lambda_pct"], Q_REF_ILLUSTRATION, h) for h in h_fin]
    ax.plot(h_fin, np.array(couts) * 100, label=classe, linewidth=1.6)
ax.set_xlabel("Horizon de cession $h$ (jours ouvrés)")
ax.set_ylabel("Coût relatif de cession $C_i(h)$ (\%, échelle log)")
ax.set_yscale("log")
ax.set_title("Courbes de coût de cession par classe d'actif")
ax.legend(fontsize=7.5, loc="upper right", framealpha=0.9)
ax.set_xlim(0, 120)
fig.tight_layout()
fig.savefig("fig_cout_par_classe.png", dpi=200)
plt.close(fig)

# Figure 2
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for nom_scen in SCENARIOS["scenario"]:
    sub = SURFACE[SURFACE["scenario"] == nom_scen].sort_values("horizon_j")
    ax.plot(sub["horizon_j"], sub["cout_portefeuille"] * 100, marker="o",
            markersize=3.5, linewidth=1.7, label=nom_scen,
            color=COULEURS_SCEN[nom_scen])
ax.set_xlabel("Horizon de cession $h$ (jours ouvrés)")
ax.set_ylabel("Coût de cession du portefeuille $C_P(h)$ (\%)")
ax.legend(fontsize=8.5, loc="upper right", framealpha=0.9)
ax.set_xlim(0, 120)
fig.tight_layout()
fig.savefig("fig_surface_scenarios.png", dpi=200)
plt.close(fig)

# Figure 3
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(DURATION_TOTALE["horizon_j"], DURATION_TOTALE["D_L"],
        color="#c0392b", linewidth=2.0, marker="o", markersize=3.5,
        label="Duration totale $D_L^r(h)$")
ax.plot(DURATION_VOLUME["horizon_j"], DURATION_VOLUME["D_L"],
        color="#2980b9", linewidth=1.5, linestyle="--",
        label="Effet volume (rachats) seul")
ax.plot(DURATION_DIRECTE["horizon_j"], DURATION_DIRECTE["D_L"],
        color="#16a085", linewidth=1.5, linestyle="--",
        label="Effet direct (marché) seul")
ax.set_xlabel("Horizon de cession $h$ (jours ouvrés)")
ax.set_ylabel("Duration de liquidité $D_L^r(h)$")
ax.legend(fontsize=8.5, loc="upper right", framealpha=0.9)
ax.set_xlim(0, 120)
fig.tight_layout()
fig.savefig("fig_duration_decomposition.png", dpi=200)
plt.close(fig)

# Figure 4 (heatmap)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
pivot = SURFACE.pivot(index="scenario", columns="horizon_j", values="cout_portefeuille") * 100
pivot = pivot.reindex(["Central", "Choc de taux", "Crise de spread", "Choc combine"])
im = ax.imshow(pivot.values, aspect="auto", cmap="Reds", extent=[0, len(HORIZONS), 0, 4])
ax.set_xticks(np.arange(len(HORIZONS)) + 0.5)
ax.set_xticklabels(HORIZONS, fontsize=7.5)
ax.set_yticks(np.arange(4) + 0.5)
ax.set_yticklabels(pivot.index[::-1], fontsize=8.5)
ax.set_xlabel("Horizon de cession $h$ (jours ouvrés)")
for i in range(4):
    for j in range(len(HORIZONS)):
        val = pivot.values[3 - i, j]
        ax.text(j + 0.5, i + 0.5, f"{val:.1f}", ha="center", va="center",
                fontsize=6.5, color="white" if val > pivot.values.max() * 0.55 else "black")
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Coût de cession (\%)", fontsize=8.5)
ax.set_title("Surface horizon--scénario--coût $C_P(h)$", fontsize=10)
fig.tight_layout()
fig.savefig("fig_heatmap_surface.png", dpi=200)
plt.close(fig)

print("
4 figures generees : fig_cout_par_classe.png, fig_surface_scenarios.png,",
      "fig_duration_decomposition.png, fig_heatmap_surface.png")